# Import

In [ ]:
import os
import random
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader, Subset
from torch.cuda.amp import autocast, GradScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Hyperparameter Setting

In [ ]:
# CFG = {
#     'IMG_SIZE': 224,
#     'BATCH_SIZE': 64,
#     'EPOCHS': 10,
#     'LEARNING_RATE': 1e-4,
#     'SEED' : 42
# }

# ----------------- Config ------------------
CFG = {
    'IMG_SIZE': 224,
    'EPOCHS': 30,
    'LEARNING_RATE': 1e-4,
    'BATCH_SIZE': 64,
    'N_FOLDS': 5,
    'SEED': 42,
    'LABEL_SMOOTHING': 0.1,
    'EARLY_STOPPING_PATIENCE': 7,
    'USE_CUTMIX': True,
    'USE_TTA': True,
    'NUM_CLASSES': 396,
    'WEIGHT_DECAY': 1e-4
}

# Fixed RandomSeed

In [ ]:
# ------------- Seed 고정 ---------------
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
seed_everything(CFG['SEED'])

# CustomDataset

In [ ]:
from PIL import Image
from torch.utils.data import Dataset

class CustomImageDataset(Dataset):
    def __init__(self, root_dir, transform=None, is_test=False):
        self.root_dir = root_dir
        self.transform = transform
        self.is_test = is_test
        self.samples = []
        self.classes = []

        if is_test:
            for fname in sorted(os.listdir(root_dir)):
                if fname.lower().endswith(('.jpg')):
                    img_path = os.path.join(root_dir, fname)
                    self.samples.append((img_path,))
        else:
            self.classes = sorted(os.listdir(root_dir))
            self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}

            for cls_name in self.classes:
                cls_folder = os.path.join(root_dir, cls_name)
                for fname in os.listdir(cls_folder):
                    if fname.lower().endswith(('.jpg')):
                        img_path = os.path.join(cls_folder, fname)
                        label = self.class_to_idx[cls_name]
                        self.samples.append((img_path, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]

        img_path = sample[0]
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        if self.is_test:
            return image  # 테스트셋은 라벨 없음
        else:
            label = sample[1]
            return image, label


# Data Load

In [ ]:
# !unzip -q /kaggle/input/hecto-ai.zip -d /kaggle/working/
# train_root = '/kaggle/working/hecto-ai/train'

In [ ]:
# import os

# for dirname, _, filenames in os.walk('/kaggle/input'):
#     print(dirname)

In [ ]:
train_root = '/kaggle/input/train'
test_root = '/kaggle/input/test'

# ✅ 전체 학습 데이터셋 생성 후 클래스 목록 추출
full_dataset = CustomImageDataset(train_root, transform=None, is_test=False)
class_names = full_dataset.classes  # ⬅️ 여기서 정의해야 이후 사용 가능

# ✅ class_to_idx 생성 (클래스 이름 → 인덱스 매핑)
class_to_idx = {cls_name: i for i, cls_name in enumerate(class_names)}

# ✅ 예시: 테스트셋 결과 출력용 샘플 수집 (optional)
samples = []
for cls_name in class_names:
    cls_folder = os.path.join(train_root, cls_name)
    for fname in os.listdir(cls_folder):
        if fname.lower().endswith('.jpg'):
            img_path = os.path.join(cls_folder, fname)
            samples.append((img_path, class_to_idx[cls_name]))


In [ ]:
# train_transform = transforms.Compose([
#     transforms.Resize((CFG['IMG_SIZE'], CFG['IMG_SIZE'])),
#     transforms.ToTensor(),
#     transforms.Normalize(mean=[0.485, 0.456, 0.406],
#                          std=[0.229, 0.224, 0.225])
# ])

# val_transform = transforms.Compose([
#     transforms.Resize((CFG['IMG_SIZE'], CFG['IMG_SIZE'])),
#     transforms.ToTensor(),
#     transforms.Normalize(mean=[0.485, 0.456, 0.406],
#                          std=[0.229, 0.224, 0.225])
# ])

# ---------- Transform 정의 ----------
train_transform = transforms.Compose([
    transforms.Resize((CFG['IMG_SIZE'] + 32, CFG['IMG_SIZE'] + 32)),
    transforms.RandomResizedCrop(CFG['IMG_SIZE'], scale=(0.8, 1.0), ratio=(0.9, 1.1)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.01),
    transforms.RandomRotation(5),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
val_transform = transforms.Compose([
    transforms.Resize((CFG['IMG_SIZE'], CFG['IMG_SIZE'])),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])


# Model Define

In [ ]:
# class BaseModel(nn.Module):
#     def __init__(self, num_classes):
#         super(BaseModel, self).__init__()
#         self.backbone = models.resnet18(pretrained=False)  # ResNet18 모델 불러오기
#         self.feature_dim = self.backbone.fc.in_features 
#         self.backbone.fc = nn.Identity()  # feature extractor로만 사용
#         self.head = nn.Linear(self.feature_dim, num_classes)  # 분류기

#     def forward(self, x):
#         x = self.backbone(x)       
#         x = self.head(x) 
#         return x

import torch.nn as nn
import torchvision.models as models

class BaseModel(nn.Module):
    def __init__(self, num_classes):
        super(BaseModel, self).__init__()
        self.backbone = models.resnet50(weights=None)  # models. 붙여야 함
        self.feature_dim = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()
        self.dropout = nn.Dropout(0.5)
        self.bn = nn.BatchNorm1d(self.feature_dim)
        self.head = nn.Linear(self.feature_dim, num_classes)

    def forward(self, x):
        x = self.backbone(x)
        x = self.dropout(x)
        x = self.bn(x)
        x = self.head(x)
        return x


In [ ]:
# ✅ full_dataset은 처음 한 번만 정의
full_dataset = CustomImageDataset(train_root, transform=None, is_test=False)
class_names = full_dataset.classes
samples = full_dataset.samples
labels = [label for _, label in samples]
targets = labels  # 👈 이거 꼭 필요함!

# ✅ TransformWrapper 정의 (변경 없음)
class TransformWrapper(Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform

    def __getitem__(self, idx):
        image, label = self.subset[idx]

        # ✅ image는 PIL → transform
        image = self.transform(image)

        # ✅ label을 torch.tensor로 변환
        label = torch.tensor(label)

        return image, label

    def __len__(self):
        return len(self.subset)


# ✅ KFold 루프
skf = StratifiedKFold(n_splits=CFG['N_FOLDS'], shuffle=True, random_state=CFG['SEED'])

for fold, (train_idx, val_idx) in enumerate(skf.split(samples, labels)):
    print(f"\n🔁 Fold {fold+1}/{CFG['N_FOLDS']}")

    train_raw = Subset(full_dataset, train_idx)
    val_raw = Subset(full_dataset, val_idx)

    train_dataset = TransformWrapper(train_raw, train_transform)
    val_dataset = TransformWrapper(val_raw, val_transform)

    ...
    # 이후는 동일 (train_loader, val_loader 등)


# Train/ Validation

In [ ]:
# ✅ 기본 라이브러리
import os
import random
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

# ✅ PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from torch.cuda.amp import autocast
from torch.amp import GradScaler  # 최신 방식

# ✅ torchvision
import torchvision.models as models
from torchvision.models import resnet50
import torchvision.transforms as transforms

# ✅ sklearn
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss


def cutmix_collate_fn(batch):
    images, labels = zip(*batch)
    batch_size = len(images)

    images = torch.stack(images)
    labels = torch.tensor(labels, dtype=torch.long)  # 확실히 텐서화

    indices = torch.randperm(batch_size)
    shuffled_images = images[indices]
    shuffled_labels = labels[indices]

    lam = np.random.beta(1.0, 1.0)
    bbx1, bby1, bbx2, bby2 = rand_bbox(images.size(), lam)
    images[:, :, bbx1:bbx2, bby1:bby2] = shuffled_images[:, :, bbx1:bbx2, bby1:bby2]

    lam = 1 - ((bbx2 - bbx1) * (bby2 - bby1) / (images.size(-1) * images.size(-2)))
    lam = torch.tensor(lam, dtype=torch.float)  # ✅ 여기가 핵심!

    return images, (labels, shuffled_labels, lam)


def rand_bbox(size, lam):
    W = size[2]
    H = size[3]
    cut_rat = np.sqrt(1. - lam)
    cut_w = int(W * cut_rat)
    cut_h = int(H * cut_rat)

    cx = np.random.randint(W)
    cy = np.random.randint(H)

    bbx1 = np.clip(cx - cut_w // 2, 0, W)
    bby1 = np.clip(cy - cut_h // 2, 0, H)
    bbx2 = np.clip(cx + cut_w // 2, 0, W)
    bby2 = np.clip(cy + cut_h // 2, 0, H)

    return bbx1, bby1, bbx2, bby2

def default_collate_fn(batch):
    images, labels = zip(*batch)
    images = torch.stack(images)
    labels = torch.tensor(labels, dtype=torch.long)
    return images, labels



skf = StratifiedKFold(n_splits=CFG['N_FOLDS'], shuffle=True, random_state=CFG['SEED'])

for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(targets)), targets)):
    print(f"\n🔁 Fold {fold + 1}/{CFG['N_FOLDS']}")

    # ✅ TransformWrapper 사용
    train_raw = Subset(full_dataset, train_idx)
    val_raw = Subset(full_dataset, val_idx)

    train_dataset = TransformWrapper(train_raw, train_transform)
    val_dataset = TransformWrapper(val_raw, val_transform)

    train_loader = DataLoader(
    train_dataset,
    batch_size=CFG['BATCH_SIZE'],
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    collate_fn=cutmix_collate_fn if CFG['USE_CUTMIX'] else default_collate_fn
)

    val_loader = DataLoader(
        val_dataset,
        batch_size=CFG['BATCH_SIZE'],
        shuffle=False,
        num_workers=4,
        pin_memory=True
    )

    model = BaseModel(num_classes=len(class_names)).to(device)
    best_logloss = float('inf')
    patience_counter = 0

    criterion = nn.CrossEntropyLoss(label_smoothing=CFG['LABEL_SMOOTHING'])
    optimizer = optim.Adam(model.parameters(), lr=CFG['LEARNING_RATE'], weight_decay=CFG['WEIGHT_DECAY'])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG['EPOCHS'])
    scaler = GradScaler()

    for epoch in range(CFG['EPOCHS']):
        print(f"\n📘 Epoch {epoch+1}/{CFG['EPOCHS']} (Fold {fold+1})")

        model.train()
        train_loss = 0.0
        for images, labels in tqdm(train_loader, desc=f"[Fold {fold+1} Epoch {epoch+1}] Training"):
            images = images.to(device)

            if isinstance(labels, tuple) and all(isinstance(x, torch.Tensor) for x in labels):
                targets1, targets2, lam = labels
                targets1 = targets1.to(device)
                targets2 = targets2.to(device)
                lam = lam.to(device)
            else:
                # 리스트일 경우 tensor로 변환
                if isinstance(labels, list):
                    if isinstance(labels[0], torch.Tensor):
                        # ✅ 리스트 안에 단일 텐서 (ex: [tensor([1,2,3])])
                        if labels[0].ndim == 1:
                            labels = labels[0]
                        else:
                            labels = torch.stack(labels)
                    else:
                        labels = torch.tensor(labels)
                labels = labels.to(device)


            optimizer.zero_grad()
            with autocast():
                outputs = model(images)
                if isinstance(labels, tuple):
                    loss = lam * criterion(outputs, targets1) + (1 - lam) * criterion(outputs, targets2)
                else:
                    loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item()

        avg_train_loss = train_loss / len(train_loader)

        # ---------- Validation ----------
        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0
        all_probs = []
        all_labels = []

        with torch.no_grad():
            for images, labels in tqdm(val_loader, desc=f"[Fold {fold+1} Epoch {epoch+1}] Validation"):
                images = images.to(device)
                labels = labels.to(device)
                with autocast():
                    outputs = model(images)
                    loss = criterion(outputs, labels)

                val_loss += loss.item()
                _, preds = torch.max(outputs, 1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)

                probs = F.softmax(outputs, dim=1)
                probs = torch.clamp(probs, 1e-7, 1 - 1e-7)
                all_probs.extend(probs.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        avg_val_loss = val_loss / len(val_loader)
        val_accuracy = 100 * correct / total
        val_logloss = log_loss(all_labels, all_probs, labels=list(range(len(class_names))))

        print(f"✅ Train Loss : {avg_train_loss:.4f} | Valid Loss : {avg_val_loss:.4f} | Valid Acc : {val_accuracy:.2f}% | LogLoss : {val_logloss:.4f}")

        if val_logloss < best_logloss:
            best_logloss = val_logloss
            torch.save(model.state_dict(), f'best_model_fold{fold+1}.pth')
            print(f"📂 Best model saved for Fold {fold+1} at Epoch {epoch+1} (LogLoss: {val_logloss:.4f})")
            patience_counter = 0
        else:
            patience_counter += 1
            print(f"⏳ Patience: {patience_counter}/{CFG['EARLY_STOPPING_PATIENCE']}")
            if patience_counter >= CFG['EARLY_STOPPING_PATIENCE']:
                print("⛔ Early stopping triggered.")
                break

        scheduler.step()


In [ ]:
# print(type(labels), type(labels[0]))  # tensor인지 확인

In [ ]:
# print(labels)

# Inference

In [ ]:
test_dataset = CustomImageDataset(test_root, transform=val_transform, is_test=True)
test_loader = DataLoader(test_dataset, batch_size=CFG['BATCH_SIZE'], shuffle=False)

In [ ]:
import torch
import torch.nn.functional as F
import pandas as pd
from tqdm import tqdm
from PIL import Image
import numpy as np

# 테스트셋 로드
test_dataset = CustomImageDataset(test_root, transform=val_transform, is_test=True)
test_loader = DataLoader(test_dataset, batch_size=CFG['BATCH_SIZE'], shuffle=False)

# 모델 로드
model = BaseModel(num_classes=len(class_names))
model.load_state_dict(torch.load('best_model.pth', map_location=device))
model.to(device)
model.eval()

# ✅ TTA 추론 함수
def predict_with_tta(model, test_loader):
    model.eval()
    final_probs = []

    with torch.no_grad():
        for imgs in test_loader:
            imgs = imgs.to(device)
            preds = []
            for _ in range(4):
                augmented = transforms.functional.hflip(imgs) if _ % 2 == 0 else imgs
                outputs = model(augmented)
                probs = F.softmax(outputs, dim=1)
                preds.append(probs)
            avg_probs = torch.stack(preds).mean(dim=0)
            final_probs.append(avg_probs.cpu())

    return torch.cat(final_probs)

# ✅ 앙상블 추론
def ensemble_inference(test_loader):
    total_probs = []
    for fold in range(CFG['N_FOLDS']):
        model = BaseModel(CFG['NUM_CLASSES'])
        model.load_state_dict(torch.load(f"best_model_fold{fold}.pth", map_location=device))
        model = model.to(device)

        if CFG['USE_TTA']:
            fold_probs = predict_with_tta(model, test_loader)
        else:
            model.eval()
            fold_probs = []
            with torch.no_grad():
                for imgs in test_loader:
                    imgs = imgs.to(device)
                    outputs = model(imgs)
                    probs = F.softmax(outputs, dim=1)
                    fold_probs.append(probs.cpu())
            fold_probs = torch.cat(fold_probs)

        total_probs.append(fold_probs)

    return torch.stack(total_probs).mean(dim=0)  # 평균

# ✅ 앙상블 + TTA 결과 추론
results = ensemble_inference(test_loader)

sample = pd.read_csv("/kaggle/input/hecto-ai/sample_submission.csv")
pred = pd.DataFrame(results.numpy(), columns=sample.columns[1:])  # ID 제외한 컬럼 정확히 일치
pred.insert(0, "ID", sample["ID"])
pred.to_csv("submission.csv", index=False, encoding="utf-8-sig")

# Submission

In [ ]:
# pred 컬럼 순서를 sample_submission 기준으로 재정렬
submission = pd.read_csv('/kaggle/input/sample_submission.csv', encoding='utf-8-sig')

class_columns = [col for col in submission.columns if col != 'ID']  # 'ID' 제외
pred = pred[class_columns]  # 예측 결과도 그 순서로 정렬

submission[class_columns] = pred.values
submission.to_csv('submission.csv', index=False, encoding='utf-8-sig')